In [6]:
# Import packages
import requests
import io
import pandas as pd
import seaborn as sbn
import matplotlib.pyplot as plt
from datetime import date, timedelta
from IPython.display import display, Markdown

In [7]:
###################
#SET E-MAIL HEADER#
###################

# This cell will serve as a header in your email. You can add some information here that may be useful for providing context. 
# If you want to change the actual text that appears, feel free to edit the "md_text" variable directly.

# Input a title of your choosing here.
title = "Open Data CLE Trend Analysis"

# Write a brief description about the analysis.
description = """
This will help you provide robust analyses on trends from Open Data CLE!
"""

# Get today's date
current_date = date.today()

# Print markdown header
md_text = f"""
## {title}
**Description**
{description}
**Data as of**  
{current_date}
"""

# Render it in the output
display(Markdown(md_text))


## Open Data CLE Trend Analysis
**Description**

This will help you provide robust analyses on trends from Open Data CLE!

**Data as of**  
2026-09-15


In [14]:
###########
#PULL DATA#
###########

# Put the URL for your API request here. You can do this using the query builder in ArcGIS Online.
url = "https://services3.arcgis.com/dty2kHktVXHrqO8i/arcgis/rest/services/Data_311/FeatureServer/0/query"

# Type out a where clause here. 
# You can utilize an "f string" to make this filter dynamic.
# NOTE: For many feature layers, the maximum amount of records the ArcGIS Online API can query is 2,000. You'll need to perform multiple queries if you are reading in more than 2k records.
where_clause = """
service_name = 'Illegal Dumping'
"""

# Set query parameters. Nothing here for you to do.
query_params = {
    "where": where_clause,       # The where clause from above.
    "returnGeometry": "false",    # We're not doing any work with spatial data. But if you want to make maps with your data, set to 'true'.
    "f": "json"                  # Tells the server to respond with JSON format.
}

## SUBMIT AND PARSE API REQUEST(S)
# Since in many cases we are limited to reading 2,000 records per query, this function will make several API requests to get all the records.
# It will also parse the request and extract the data into a list of records.
def get_data(url,query_params):
    """
    This function retrieves data from ArcGIS Online FeatureLayer via a series of GET requests. 
    It will pull data in groups of 2,000 records, and append all the data to one list of records.

    Args:
        url (str): Spark session context.
        query_params (dict): A spark DataFrame to geocode.

    Returns:
        list: The spark DataFrame, with new geocoded columns appended.
    """
    # Get record count
    record_count = requests.get(url,params={"where":query_params['where'],"returnCountOnly":"true","f":"json"}).json()['count']

    # Split record count into offsets
    offsets = range(0,record_count,2000)
    
    # List of data records
    results = []
    
    # Loop through offsets and get data for each offset
    for i,offset in enumerate(offsets):
        # Perform a GET request with the given offset
        query_params['resultOffset'] = offset
        req = requests.get(url,params=query_params)
        # Get JSON
        resp = req.json()

        # Extract data and append it to our final result
        data = [a['attributes'] for a in resp['features']]

        results += data
    
    # Ensure the number of records matches the record count of the Feature Layer.
    assert len(results) == record_count

    return results

data = get_data(url, query_params)


In [16]:
######################
#CONVERT TO DATAFRAME#
######################
# If you use the get_data function from above, your data should look something like this:.
"""
[{'service_request_id': '202000403109',
  'service_category': 'Trash & Recycling',
  'service_name': 'Waste Cart Concerns'},
 {'service_request_id': '202000403083',
  'service_category': 'Building & Housing',
  'service_name': 'Electrical Issue'},
 {'service_request_id': '202000403082',
  'service_category': 'Street Issues',
  'service_name': 'Debris in Street'}]
"""

# The format above is known as "records" format, and will allow you to automatically convert to a pandas dataframe when doing pd.DataFrame(records).
# Try converting to DataFrame below:
df = pd.DataFrame(data)

df

,OBJECTID,service_request_id,address,service_category,service_name,agency_responsible,division_responsible,status_description,requested_datetime,updated_datetime,...,parcelpin,neighborhood,ward_name,ward,ward_name_2014,ward_2014,ward_name_2026,ward_2026,lat,long
0,2,202000405340,"1412 E 80th St, Cleveland, OH 44103, US",Illegal Dumping,Illegal Dumping,Public Works,Streets,Open,1789429621000,1789429627000,...,10605028,Hough,Ward 8,8.0,Ward 7,7.0,Ward 8,8.0,41.516762,-81.633278
1,5,202000405337,"5103 Luther Ave, Cleveland, OH 44103, US",Illegal Dumping,Illegal Dumping,Public Works,Streets,Open,1789429002000,1789429006000,...,10422036,Goodrich-Kirtland Pk,Ward 8,8.0,Ward 7,7.0,Ward 8,8.0,41.515868,-81.654459
2,47,202000405295,"11013 Mount Overlook Ave, Cleveland, OH 44104, US",Illegal Dumping,Illegal Dumping,Public Works,Streets,Open,1789418903000,1789418906000,...,12132106,Buckeye-Woodhill,Ward 6,6.0,Ward 6,6.0,Ward 6,6.0,41.491471,-81.607266
3,49,202000405293,"3757 E 144th St, Cleveland, OH 44128, US",Illegal Dumping,Illegal Dumping,Public Works,Streets,Open,1789418871000,1789418873000,...,13908003,Mount Pleasant,Ward 1,1.0,Ward 2,2.0,Ward 1,1.0,41.457162,-81.581125
4,89,202000405253,"13212 Terminal Ave, Cleveland, OH 44135, US",Illegal Dumping,Illegal Dumping,Public Works,Streets,Open,1789414527000,1789414529000,...,02223073,Bellaire-Puritas,Ward 13,13.0,Ward 16,16.0,Ward 13,13.0,41.439270,-81.782422
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9698,178501,101000000381,"6527 SUPERIOR AVE, CLEVELAND, Ohio 44103",Illegal Dumping,Illegal Dumping,Public Works,Streets,Closed,1709244900000,1709564497000,...,10522025,St.Clair-Superior,Ward 8,8.0,Ward 7,7.0,Ward 8,8.0,41.519687,-81.644771
9699,178511,101000000349,"4000 E 71 ST, CLEVELAND, Ohio 44105",Illegal Dumping,Illegal Dumping,Public Works,Streets,Closed,1709242251000,1709570383000,...,NaN,Broadway-Slavic Village,Ward 2,2.0,Ward 12,12.0,Ward 2,2.0,41.449566,-81.638878
9700,178520,101000000307,"10505 ST CLAIR AVE, CLEVELAND, Ohio 44108",Illegal Dumping,Illegal Dumping,Public Works,Streets,Closed,1709238895000,1709816039000,...,10812001,Glenville,Ward 9,9.0,Ward 9,9.0,Ward 9,9.0,41.539945,-81.614967
9701,178528,101000000278,"4248 E 71 ST, CLEVELAND, Ohio 44105",Illegal Dumping,Illegal Dumping,Public Works,Streets,Closed,1709236534000,1709566764000,...,13230023,Broadway-Slavic Village,Ward 2,2.0,Ward 12,12.0,Ward 2,2.0,41.443114,-81.639155


In [22]:
##########
#ANALYSIS#
##########

# Conduct your data analysis below. You should use the dataframe from above as your starting point. Feel free to add more cells to separate output.
df['requested_datetime'] = pd.to_datetime(df['requested_datetime'], unit='ms')
datediff_7 = current_date - timedelta(days=7)
date_filter = df['requested_datetime'].dt.date >= datediff_7

illegal_dump_count = df[date_filter]['service_request_id'].count()

print(f"There have been {illegal_dump_count} cases of illegal dumping with the last 7 days")
# We import the "seaborn" package in the first cell above. You can use this or another package of your choosing for creating visualizations.
# If you choose to import additional packages, be sure to update the dependencies in your GitHub Action! Otherwise the workflow will fail.

There have been 90 cases of illegal dumping with the last 7 days
